# Merge and Prepare the Dudley Datasets

## Purpose
Combine the five cleaned datasets to explore local factors associated
with economic activity and possible employment support needs in Dudley.

## Objectives
- Verify geographic compatibility between the datasets.
- Select relevant columns and distinguish their source years.
- Merge compatible areas using LSOA codes.
- Check merge coverage, missing values and duplicate records.
- Calculate percentages using the appropriate population or
  household denominator from each source.
- Save the combined dataset for later analysis.

## Inputs
Five cleaned CSV files from `cleaned_datasets`:
- `dudley_economic_activity_cleaned.csv`
- `dudley_qualifications_cleaned.csv`
- `dudley_disability_16_cleaned.csv`
- `dudley_car_availability_cleaned.csv`
- `dudley_iod_cleaned.csv`

## Planned Output
`cleaned_datasets/dudley_combined_prepared.csv`

## Geographic and Time Coverage
The Census datasets contain 203 Dudley areas. The deprivation
dataset contains 201 areas using 2011 geography.

The datasets share 199 LSOA codes. The planned merge will use
matching areas after boundary compatibility has been checked.
Any exclusions will be documented.

Deprivation data comes from the 2019 release and provides historical
context alongside Census 2021 data.

## Ethical Considerations
Area-level patterns do not establish individual circumstances or
prove causes of unemployment. Disability, qualifications and car
availability must be interpreted respectfully and in context.

Different source dates, population definitions and excluded areas
will be considered when interpreting results. The original cleaned
datasets will remain unchanged.

## Additional Comments
ChatGPT assisted with planning, code development and documentation.
This notebook covers merging and preparation; EDA, hypothesis
testing and modelling will follow separately.

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [2]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [3]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

## 1. Load the Cleaned Datasets

Load each cleaned CSV into a separate dataframe.
Display their dimensions and column names to confirm the available
fields before selecting variables or calculating percentages.

No source files will be changed.

In [5]:
# Developed with ChatGPT assistance to load the cleaned datasets.
import pandas as pd

# Map each dataset to its saved CSV filename.
dataset_files = {
    "economic": "dudley_economic_activity_cleaned.csv",
    "qualifications": "dudley_qualifications_cleaned.csv",
    "disability": "dudley_disability_16_cleaned.csv",
    "car": "dudley_car_availability_cleaned.csv",
    "iod": "dudley_iod_cleaned.csv"
}

# Load each file separately, preserving the saved source files.
datasets = {
    name: pd.read_csv(f"cleaned_datasets/{filename}")
    for name, filename in dataset_files.items()
}

# Inspect dimensions and exact column names before merging.
for name, dataframe in datasets.items():
    print(f"\n{name.upper()}")
    print(f"Rows: {dataframe.shape[0]}, Columns: {dataframe.shape[1]}")
    print(dataframe.columns.tolist())


ECONOMIC
Rows: 203, Columns: 11
['Original_Area', 'Total_Aged_16_Plus', 'Economically_Active_Excl_FT_Students', 'Employed_Excl_FT_Students', 'Unemployed_Excl_FT_Students', 'Inactive_Student', 'Inactive_Home_Or_Family', 'Inactive_Long_Term_Sick_Or_Disabled', 'Inactive_Other', 'LSOA_Code', 'LSOA_Name']

QUALIFICATIONS
Rows: 203, Columns: 10
['LSOA_Name', 'LSOA_Code', 'Total_Aged_16_Plus', 'No_Qualifications', 'Level_1_And_Entry', 'Level_2', 'Apprenticeship', 'Level_3', 'Level_4_And_Above', 'Other_Qualifications']

DISABILITY
Rows: 203, Columns: 7
['LSOA_Name', 'LSOA_Code', 'Total_16_Plus', 'Disabled_16_Plus', 'Limited_A_Lot_16_Plus', 'Limited_A_Little_16_Plus', 'Not_Disabled_16_Plus']

CAR
Rows: 203, Columns: 7
['LSOA_Name', 'LSOA_Code', 'Total_Households', 'No_Car_Van_Households', 'One_Car_Van_Households', 'Two_Cars_Vans_Households', 'Three_Plus_Cars_Vans_Households']

IOD
Rows: 201, Columns: 57
['LSOA_code_2011', 'LSOA_name_2011', 'Local_Authority_District_code_2019', 'Local_Authority

### Findings
- All five cleaned datasets loaded successfully.
- Economic activity contains 203 rows and 11 columns.
- Qualifications contains 203 rows and 10 columns.
- Disability and car availability each contain 203 rows and 7 columns.
- Deprivation contains 201 rows and 57 columns.
- The four Census datasets use `LSOA_Code`; deprivation uses
  `LSOA_code_2011`.
- Population totals have overlapping names across datasets.
  They must remain distinguishable so percentages use the
  correct source-specific denominator.
- The economic activity file contains selected categories,
  including employment measures excluding full-time students.
  These must not be labelled as covering all employment.
- No datasets have been merged or modified.

---

## 2. Check Merge Keys and Shared Coverage

Check that each dataset has complete, unique LSOA codes so merging
will not create duplicate area records.

Confirm whether the four Census datasets contain the same codes
and count the codes shared by all five datasets.

Matching codes establish join coverage. Compatibility between
2011 and 2021 boundaries still requires a separate check.

In [8]:
# Developed with ChatGPT assistance to check merge keys and coverage.

# Identify the geographic key used by each dataset.
key_columns = {
    name: "LSOA_code_2011" if name == "iod" else "LSOA_Code"
    for name in datasets
}

# Check missing and duplicate keys before merging.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html
check_results = []

for name, dataframe in datasets.items():
    codes = dataframe[key_columns[name]]

    check_results.append({
        "Dataset": name,
        "Rows": len(dataframe),
        "Missing_codes": codes.isna().sum(),
        "Duplicate_codes": codes.duplicated().sum()
    })

key_checks = pd.DataFrame(check_results)
display(key_checks)

# Remove missing codes and create sets of unique codes.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.dropna.html
# https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset
code_sets = {}

for name, dataframe in datasets.items():
    codes = dataframe[key_columns[name]].dropna()
    code_sets[name] = set(codes)

# Check whether the Census datasets cover the same areas.
# Reference:
# https://docs.python.org/3/library/functions.html#all
census_names = [
    "economic",
    "qualifications",
    "disability",
    "car"
]

same_census_codes = all(
    code_sets[name] == code_sets["economic"]
    for name in census_names
)

# Find codes present in every dataset.
# Reference:
# https://docs.python.org/3/library/stdtypes.html#frozenset.intersection
shared_codes = set.intersection(*code_sets.values())

print("All four Census code sets match:", same_census_codes)
print("Codes shared by all five datasets:", len(shared_codes))

,Dataset,Rows,Missing_codes,Duplicate_codes
0,economic,203,0,0
1,qualifications,203,0,0
2,disability,203,0,0
3,car,203,0,0
4,iod,201,0,0


All four Census code sets match: True
Codes shared by all five datasets: 199


### Findings
- All five datasets have complete, unique LSOA codes.
- No missing or duplicate codes were found.
- The four Census datasets contain the same 203 area codes.
- A total of 199 codes are shared by all five datasets.
- The keys support one-to-one joins without duplicate matches.
- Compatibility between the 2011 and 2021 boundaries still
  needs verification before merging.
- No records have been removed or merged.

---

## 3. Load the Geographic Lookup

Load the official ONS lookup linking 2011 and 2021 LSOA areas.
Inspect its columns and preview five records before using its
change indicator to check boundary compatibility.

This lookup supports geographic validation; it will not add
analytical features to the combined dataset.

In [9]:
# Developed with ChatGPT assistance to inspect the ONS lookup.

# Load the downloaded lookup without changing its contents.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
lookup_path = "raw_datasets/lsoa_2011_2021_lookup.csv"
lookup_df = pd.read_csv(lookup_path)

print(f"Rows: {lookup_df.shape[0]}")
print(f"Columns: {lookup_df.shape[1]}")

# Display exact column names to guide the next check.
for column in lookup_df.columns:
    print(column)

# Preview five records.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html
display(lookup_df.head())

Rows: 35796
Columns: 9
LSOA11CD
LSOA11NM
LSOA21CD
LSOA21NM
CHGIND
LAD22CD
LAD22NM
LAD22NMW
ObjectId


,LSOA11CD,LSOA11NM,LSOA21CD,LSOA21NM,CHGIND,LAD22CD,LAD22NM,LAD22NMW,ObjectId
0,E01031349,Adur 001A,E01031349,Adur 001A,U,E07000223,Adur,NaN,1
1,E01031350,Adur 001B,E01031350,Adur 001B,U,E07000223,Adur,NaN,2
2,E01031351,Adur 001C,E01031351,Adur 001C,U,E07000223,Adur,NaN,3
3,E01031352,Adur 001D,E01031352,Adur 001D,U,E07000223,Adur,NaN,4
4,E01031370,Adur 001E,E01031370,Adur 001E,U,E07000223,Adur,NaN,5


### Findings
- The lookup loaded successfully with 35,796 rows and 9 columns.
- `LSOA11CD` and `LSOA21CD` contain the geographic codes
  for 2011 and 2021.
- `CHGIND` records the type of geographic change.
- The five previewed records have indicator `U` (unchanged)
  and identical codes in both years.
- These preview records are for Adur. Dudley's matching areas
  still need to be checked.

---

## 4. Verify Boundary Compatibility for Shared Areas

Check the 199 shared codes against the ONS lookup.

Each shared code should have exactly one lookup record, the same
2011 and 2021 code, and change indicator `U` (unchanged).
Report any exceptions before merging.

In [10]:
# Developed with ChatGPT assistance to verify area boundaries.

# Select lookup records for the shared 2011 codes.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html
shared_lookup = lookup_df.loc[
    lookup_df["LSOA11CD"].isin(shared_codes)
].copy()

# Check that every shared code has exactly one lookup record.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.reindex.html
lookup_counts = shared_lookup["LSOA11CD"].value_counts()
lookup_counts = lookup_counts.reindex(
    sorted(shared_codes),
    fill_value=0
)

print("Shared codes checked:", len(shared_codes))
print("Codes missing from lookup:", (lookup_counts == 0).sum())
print("Codes with multiple records:", (lookup_counts > 1).sum())

# Flag changed areas or different codes between the two years.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.eq.html
same_code = shared_lookup["LSOA11CD"].eq(
    shared_lookup["LSOA21CD"]
)
unchanged = shared_lookup["CHGIND"].eq("U")
exceptions = shared_lookup.loc[~(same_code & unchanged)]

print("Records failing boundary checks:", len(exceptions))
print("\nChange indicators:")
print(shared_lookup["CHGIND"].value_counts(dropna=False))

# Preview any exceptions, showing only the relevant fields.
display(
    exceptions[
        ["LSOA11CD", "LSOA21CD", "CHGIND", "LAD22NM"]
    ].head()
)

Shared codes checked: 199
Codes missing from lookup: 0
Codes with multiple records: 0
Records failing boundary checks: 0

Change indicators:
CHGIND
U    199
Name: count, dtype: int64


,LSOA11CD,LSOA21CD,CHGIND,LAD22NM


### Findings
- All 199 shared codes were found in the official ONS lookup.
- Each code has exactly one lookup record.
- All records have identical 2011 and 2021 LSOA codes.
- All 199 records have change indicator `U` (unchanged).
- No boundary-check exceptions were found.
- The shared areas are geographically compatible for merging.
  Differences in dataset dates still apply.

---

## 5. Check Percentage Denominators

Check that each Census dataset's population or household total
is present and greater than zero before calculating percentages.

Use the total from the same dataset as the category being measured.
Do not substitute totals from another source, even when the
population labels are similar.

The deprivation rates are already supplied as proportions and
will be handled separately.

In [11]:
# Developed with ChatGPT assistance to check denominators.

# Use each dataset's own population or household total.
denominator_columns = {
    "economic": "Total_Aged_16_Plus",
    "qualifications": "Total_Aged_16_Plus",
    "disability": "Total_16_Plus",
    "car": "Total_Households"
}

# Check missing, zero and negative denominator values.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html
# https://pandas.pydata.org/docs/reference/api/pandas.Series.le.html
denominator_results = []

for name, column in denominator_columns.items():
    totals = datasets[name][column]

    denominator_results.append({
        "Dataset": name,
        "Denominator": column,
        "Minimum": totals.min(),
        "Maximum": totals.max(),
        "Missing": totals.isna().sum(),
        "Zero_or_negative": totals.le(0).sum()
    })

denominator_checks = pd.DataFrame(denominator_results)
display(denominator_checks)

,Dataset,Denominator,Minimum,Maximum,Missing,Zero_or_negative
0,economic,Total_Aged_16_Plus,986,2166,0,0
1,qualifications,Total_Aged_16_Plus,985,2169,0,0
2,disability,Total_16_Plus,985,2168,0,0
3,car,Total_Households,460,1183,0,0


### Findings
- All four denominator columns contain positive values.
- No missing, zero or negative denominators were found.
- Population totals vary slightly between the Census datasets.
  Each percentage will therefore use its own source's total.
- Car availability percentages will use household totals,
  rather than population totals.
- These totals support calculating shares of residents aged
  16 and over, or households, as appropriate.
- An unemployment rate would require a separate economically
  active denominator; it is not the same as a population share.

---

## 6. Calculate Qualification Percentages

Calculate each qualification category as a percentage of residents
aged 16 and over, using the qualifications dataset's own total.

Retain the original counts and add percentage columns in a separate
working copy. Keep full precision for later analysis.

In [12]:
# Developed with ChatGPT assistance to calculate qualification shares.

# Create a working copy, keeping the loaded source unchanged.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
qualifications_prepared = datasets["qualifications"].copy()

qualification_columns = [
    "No_Qualifications",
    "Level_1_And_Entry",
    "Level_2",
    "Apprenticeship",
    "Level_3",
    "Level_4_And_Above",
    "Other_Qualifications"
]

# Divide each category by its source population total.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.div.html
population = qualifications_prepared["Total_Aged_16_Plus"]

for column in qualification_columns:
    percentage_column = f"{column}_Pct"
    qualifications_prepared[percentage_column] = (
        qualifications_prepared[column].div(population) * 100
    )

# Check that all calculated percentages fall between 0 and 100.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
qualification_pct_columns = [
    f"{column}_Pct" for column in qualification_columns
]

invalid_percentages = qualifications_prepared[
    qualification_pct_columns
].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print("Percentage columns added:", len(qualification_pct_columns))
print("Values outside 0–100:", invalid_percentages.sum())

# Preview five areas; round only the displayed values.
preview_columns = ["LSOA_Name"] + qualification_pct_columns
display(
    qualifications_prepared[preview_columns].head().round(2)
)

Percentage columns added: 7
Values outside 0–100: 0


,LSOA_Name,No_Qualifications_Pct,Level_1_And_Entry_Pct,Level_2_Pct,Apprenticeship_Pct,Level_3_Pct,Level_4_And_Above_Pct,Other_Qualifications_Pct
0,Dudley 001A,26.63,11.18,17.65,6.23,17.47,18.08,2.75
1,Dudley 001B,24.17,10.43,14.77,7.74,16.90,22.04,3.95
2,Dudley 001C,28.30,11.54,17.39,5.77,14.31,18.81,3.87
3,Dudley 001D,32.70,10.99,17.09,6.16,14.78,15.48,2.80
4,Dudley 001E,32.10,12.51,15.47,5.93,14.90,16.30,2.80


### Findings
- Seven qualification percentage columns were added.
- Each percentage uses the qualifications dataset's own
  population aged 16 and over as its denominator.
- No calculated percentages fall outside 0–100.
- In Dudley 001A, 26.63% of residents aged 16 and over have
  no qualifications, while 18.08% have Level 4 or above.
- Original counts are retained in `qualifications_prepared`.
- Full precision is preserved; only the preview is rounded
  to two decimal places.
- The original cleaned CSV has not been changed.

---

## 7. Calculate Disability Percentages

Calculate disability categories as percentages of all residents
aged 16 and over, using the disability dataset's own total.

The limited-a-lot and limited-a-little percentages also use all
residents aged 16 and over, not only disabled residents.
These two categories are subsets of the disabled category.

Retain the original counts and preserve full calculation precision.

In [13]:
# Developed with ChatGPT assistance to calculate disability shares.

# Create a separate working copy.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
disability_prepared = datasets["disability"].copy()

disability_columns = [
    "Disabled_16_Plus",
    "Limited_A_Lot_16_Plus",
    "Limited_A_Little_16_Plus",
    "Not_Disabled_16_Plus"
]

# Calculate each category's share of all residents aged 16+.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.div.html
disability_population = disability_prepared["Total_16_Plus"]

for column in disability_columns:
    disability_prepared[f"{column}_Pct"] = (
        disability_prepared[column].div(disability_population) * 100
    )

# Check that percentages fall between 0 and 100.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
disability_pct_columns = [
    f"{column}_Pct" for column in disability_columns
]

invalid_disability_pct = disability_prepared[
    disability_pct_columns
].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print("Percentage columns added:", len(disability_pct_columns))
print("Values outside 0–100:", invalid_disability_pct.sum())

# Preview five areas, rounding only the displayed values.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.round.html
preview_columns = ["LSOA_Name"] + disability_pct_columns
display(
    disability_prepared[preview_columns].head().round(2)
)

Percentage columns added: 4
Values outside 0–100: 0


,LSOA_Name,Disabled_16_Plus_Pct,Limited_A_Lot_16_Plus_Pct,Limited_A_Little_16_Plus_Pct,Not_Disabled_16_Plus_Pct
0,Dudley 001A,21.34,9.39,11.95,78.66
1,Dudley 001B,20.65,9.57,11.08,79.35
2,Dudley 001C,23.34,11.44,11.91,76.66
3,Dudley 001D,26.33,14.57,11.76,73.67
4,Dudley 001E,22.92,11.21,11.71,77.08


### Findings
- Four disability percentage columns were added.
- All percentages use the disability dataset's total population
  aged 16 and over.
- No calculated percentages fall outside 0–100.
- In Dudley 001A, 21.34% of residents aged 16 and over are
  disabled: 9.39% have activities limited a lot and 11.95%
  have activities limited a little.
- The limitation categories are subsets of the disabled category
  and must not be added to it as separate groups.
- Original counts and full calculation precision are retained.
  Only the preview is rounded to two decimal places.
- The original cleaned CSV has not been changed.

---

## 8. Calculate Car Availability Percentages

Calculate each car or van availability category as a percentage
of households, using the car availability dataset's own total.

These percentages describe households, not individual residents.
Having no car or van does not by itself establish an inability
to travel to work.

Retain the original counts and full calculation precision.

In [14]:
# Developed with ChatGPT assistance to calculate household shares.

# Create a separate working copy.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
car_prepared = datasets["car"].copy()

car_columns = [
    "No_Car_Van_Households",
    "One_Car_Van_Households",
    "Two_Cars_Vans_Households",
    "Three_Plus_Cars_Vans_Households"
]

# Calculate each category's share of all households.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.div.html
household_total = car_prepared["Total_Households"]

for column in car_columns:
    car_prepared[f"{column}_Pct"] = (
        car_prepared[column].div(household_total) * 100
    )

# Check that percentages fall between 0 and 100.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
car_pct_columns = [
    f"{column}_Pct" for column in car_columns
]

invalid_car_pct = car_prepared[car_pct_columns].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print("Percentage columns added:", len(car_pct_columns))
print("Values outside 0–100:", invalid_car_pct.sum())

# Preview five areas, rounding only the displayed values.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.round.html
preview_columns = ["LSOA_Name"] + car_pct_columns
display(
    car_prepared[preview_columns].head().round(2)
)

Percentage columns added: 4
Values outside 0–100: 0


,LSOA_Name,No_Car_Van_Households_Pct,One_Car_Van_Households_Pct,Two_Cars_Vans_Households_Pct,Three_Plus_Cars_Vans_Households_Pct
0,Dudley 001A,29.59,39.00,23.47,7.94
1,Dudley 001B,12.03,41.77,34.02,12.18
2,Dudley 001C,23.10,41.64,26.29,8.97
3,Dudley 001D,40.78,38.59,14.68,5.95
4,Dudley 001E,23.86,43.01,24.65,8.48


### Findings
- Four car or van availability percentage columns were added.
- Each percentage uses the dataset's total households as
  its denominator.
- No calculated percentages fall outside 0–100.
- In Dudley 001A, 29.59% of households have no car or van,
  while 39.00% have one.
- These are household shares, not percentages of residents.
- Original counts and full calculation precision are retained.
  Only the preview is rounded to two decimal places.
- The original cleaned CSV has not been changed.

---

## 9. Calculate Economic Activity Population Shares

Calculate the selected economic activity categories as percentages
of all residents aged 16 and over, using this dataset's own total.

Employment, unemployment and economic activity counts exclude
full-time students, as indicated in their column names.

The unemployment percentage calculated here is a share of all
residents aged 16 and over, not an unemployment rate.

The available inactivity categories do not cover every reason
for inactivity, so they will not be summed into a total.
Some columns overlap and must not be added together.

In [15]:
# Developed with ChatGPT assistance to calculate population shares.

# Create a separate working copy.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
economic_prepared = datasets["economic"].copy()

economic_columns = [
    "Economically_Active_Excl_FT_Students",
    "Employed_Excl_FT_Students",
    "Unemployed_Excl_FT_Students",
    "Inactive_Student",
    "Inactive_Home_Or_Family",
    "Inactive_Long_Term_Sick_Or_Disabled",
    "Inactive_Other"
]

# Express each selected category as a share of all residents 16+.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.div.html
economic_population = economic_prepared["Total_Aged_16_Plus"]

for column in economic_columns:
    economic_prepared[f"{column}_Pct"] = (
        economic_prepared[column].div(economic_population) * 100
    )

# Check that percentages fall between 0 and 100.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
economic_pct_columns = [
    f"{column}_Pct" for column in economic_columns
]

invalid_economic_pct = economic_prepared[
    economic_pct_columns
].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print("Percentage columns added:", len(economic_pct_columns))
print("Values outside 0–100:", invalid_economic_pct.sum())

# Preview five areas, rounding only the displayed values.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.round.html
preview_columns = ["LSOA_Name"] + economic_pct_columns
display(
    economic_prepared[preview_columns].head().round(2)
)

Percentage columns added: 7
Values outside 0–100: 0


,LSOA_Name,Economically_Active_Excl_FT_Students_Pct,Employed_Excl_FT_Students_Pct,Unemployed_Excl_FT_Students_Pct,Inactive_Student_Pct,Inactive_Home_Or_Family_Pct,Inactive_Long_Term_Sick_Or_Disabled_Pct,Inactive_Other_Pct
0,Dudley 001A,59.25,54.25,5.01,5.93,10.02,4.46,4.28
1,Dudley 001B,60.05,57.83,2.22,2.77,3.96,2.77,2.45
2,Dudley 001C,59.13,55.26,3.87,2.77,5.45,5.22,2.21
3,Dudley 001D,48.99,43.19,5.80,5.45,8.32,10.34,4.12
4,Dudley 001E,54.70,51.48,3.21,3.38,6.01,5.68,4.12


### Findings
- Seven economic activity percentage columns were added.
- Each percentage uses the economic activity dataset's total
  population aged 16 and over.
- No calculated percentages fall outside 0–100.
- In Dudley 001A, employed residents excluding full-time students
  represent 54.25% of the population aged 16 and over.
- Unemployed residents excluding full-time students represent
  5.01% of that population. This is a population share,
  not an unemployment rate.
- No overall inactivity percentage was calculated because the
  available inactivity categories are incomplete.
- Original counts and full calculation precision are retained.
  Only the preview is rounded to two decimal places.
- The original cleaned CSV has not been changed.

---

## 10. Prepare Columns for Merging

Keep one LSOA code column for joining and retain the area name
from the economic activity dataset.

Give population totals distinct names so their sources remain
clear. Prefix deprivation measures with `IoD2019_` to identify
their release year.

Remove repeated area-name fields and the original economic area
label from the merge copies. Preserve the prepared source tables.

In [16]:
# Developed with ChatGPT assistance to prepare merge columns.

# Remove redundant labels and distinguish population totals.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
economic_merge = economic_prepared.drop(
    columns=["Original_Area"]
).rename(columns={
    "Total_Aged_16_Plus": "Economic_Total_16_Plus"
})

qualifications_merge = qualifications_prepared.drop(
    columns=["LSOA_Name"]
).rename(columns={
    "Total_Aged_16_Plus": "Qualifications_Total_16_Plus"
})

disability_merge = disability_prepared.drop(
    columns=["LSOA_Name"]
).rename(columns={
    "Total_16_Plus": "Disability_Total_16_Plus"
})

car_merge = car_prepared.drop(columns=["LSOA_Name"])

# Standardise the deprivation join key and label its release year.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
iod_merge = datasets["iod"].drop(columns=["LSOA_name_2011"])

iod_names = {
    column: (
        "LSOA_Code"
        if column == "LSOA_code_2011"
        else f"IoD2019_{column}"
    )
    for column in iod_merge.columns
}
iod_merge = iod_merge.rename(columns=iod_names)

# Collect the merge copies and check for overlapping field names.
merge_tables = {
    "economic": economic_merge,
    "qualifications": qualifications_merge,
    "disability": disability_merge,
    "car": car_merge,
    "iod": iod_merge
}

# Only LSOA_Code should appear in more than one merge table.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html
all_columns = [
    column
    for dataframe in merge_tables.values()
    for column in dataframe.columns
    if column != "LSOA_Code"
]

column_counts = pd.Series(all_columns).value_counts()
overlapping_columns = column_counts[column_counts > 1]

print("Overlapping non-key column names:", len(overlapping_columns))
print(overlapping_columns.to_string())

for name, dataframe in merge_tables.items():
    print(
        f"{name}: {dataframe.shape[0]} rows, "
        f"{dataframe.shape[1]} columns"
    )

Overlapping non-key column names: 0
Series([], )
economic: 203 rows, 17 columns
qualifications: 203 rows, 16 columns
disability: 203 rows, 10 columns
car: 203 rows, 10 columns
iod: 201 rows, 56 columns


### Findings
- All five merge tables use `LSOA_Code` as their join key.
- No overlapping non-key column names were found.
- Population totals have distinct source-specific names.
- Deprivation fields carry the `IoD2019_` prefix.
- The economic activity table retains the area name;
  redundant area labels were removed from the merge copies.
- All source rows remain available: 203 in each Census table
  and 201 in the deprivation table.
- No merging or geographic exclusions have taken place yet.

---

## 11. Merge the Five Datasets

Join the prepared tables using `LSOA_Code`, retaining only areas
present in every dataset.

Validate each join as one-to-one to prevent duplicate area matches.
The expected result is 199 areas whose unchanged boundaries have
already been confirmed through the ONS lookup.

Record row counts after each join and check the final code coverage.
The separate source tables remain unchanged.

In [17]:
# Developed with ChatGPT assistance to merge the prepared datasets.

# Start with a copy of the economic activity table.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html
combined_df = economic_merge.copy()
merge_results = []

# Retain matching codes and enforce one-to-one joins.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html
merge_order = ["qualifications", "disability", "car", "iod"]

for name in merge_order:
    rows_before = len(combined_df)

    combined_df = combined_df.merge(
        merge_tables[name],
        on="LSOA_Code",
        how="inner",
        validate="one_to_one"
    )

    merge_results.append({
        "Dataset_added": name,
        "Rows_before": rows_before,
        "Rows_after": len(combined_df),
        "Rows_excluded": rows_before - len(combined_df)
    })

display(pd.DataFrame(merge_results))

# Confirm the merged codes match the verified shared set.
# Reference:
# https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset
merged_codes = set(combined_df["LSOA_Code"])

print("Final rows:", combined_df.shape[0])
print("Final columns:", combined_df.shape[1])
print("Codes match verified shared areas:", merged_codes == shared_codes)

# Preview five area identifiers.
display(combined_df[["LSOA_Code", "LSOA_Name"]].head())

,Dataset_added,Rows_before,Rows_after,Rows_excluded
0,qualifications,203,203,0
1,disability,203,203,0
2,car,203,203,0
3,iod,203,199,4


Final rows: 199
Final columns: 105
Codes match verified shared areas: True


,LSOA_Code,LSOA_Name
0,E01009763,Dudley 001A
1,E01009764,Dudley 001B
2,E01009767,Dudley 001C
3,E01009769,Dudley 001D
4,E01009770,Dudley 001E


### Findings
- All four joins passed one-to-one validation.
- Joining the Census datasets retained all 203 areas.
- Adding deprivation data excluded four unmatched Census areas.
- The combined dataset contains 199 rows and 105 columns.
- Its LSOA codes exactly match the 199 shared areas whose
  unchanged boundaries were verified using the ONS lookup.
- The two deprivation-only areas were also excluded by the
  inner join.
- The combined dataset covers 199 of Dudley's 203 Census areas.
  Findings must acknowledge this reduced coverage.
- The separate source tables and cleaned CSVs remain unchanged.

---

## 12. Validate the Combined Dataset

Check the merged dataset for missing values, duplicate rows,
duplicate LSOA codes and duplicate column names.

Check that numeric values are finite and that the calculated
percentage columns remain within 0–100 after merging.

In [18]:
# Developed with ChatGPT assistance to validate the merged dataset.
import numpy as np

# Check completeness and duplicate records or identifiers.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html
print("Missing values:", combined_df.isna().sum().sum())
print("Duplicate rows:", combined_df.duplicated().sum())
print(
    "Duplicate LSOA codes:",
    combined_df["LSOA_Code"].duplicated().sum()
)
print(
    "Duplicate column names:",
    combined_df.columns.duplicated().sum()
)

# Check numeric columns for infinite values.
# References:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html
# https://numpy.org/doc/stable/reference/generated/numpy.isinf.html
numeric_data = combined_df.select_dtypes(include="number")
print("Infinite numeric values:", np.isinf(numeric_data).sum().sum())

# Check all calculated percentage columns after merging.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html
percentage_columns = [
    column for column in combined_df.columns
    if column.endswith("_Pct")
]

invalid_pct = combined_df[percentage_columns].apply(
    lambda column: (~column.between(0, 100)).sum()
)

print("Percentage columns checked:", len(percentage_columns))
print("Percentage values outside 0–100:", invalid_pct.sum())

Missing values: 0
Duplicate rows: 0
Duplicate LSOA codes: 0
Duplicate column names: 0
Infinite numeric values: 0
Percentage columns checked: 22
Percentage values outside 0–100: 0


### Findings
- No missing values or duplicate rows were found.
- LSOA codes and column names are unique.
- No infinite numeric values were found.
- All 22 calculated percentage columns were checked.
- No percentage values fall outside 0–100.
- The combined dataset passed these validation checks
  without requiring corrections.

---

# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
